## DATA Loading, CHunking and DATABASE CREAtion


In [39]:
import time
import chromadb

from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_core.documents import Document

from langchain_chroma import Chroma

from datetime import datetime


In [40]:
pdf_folder_location = "tesla-annual-reports"

In [41]:
pdf_loader = PyPDFDirectoryLoader(pdf_folder_location)

In [42]:
type(pdf_loader)

langchain_community.document_loaders.pdf.PyPDFDirectoryLoader

In [43]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap=16
)

In [44]:
tesla_10k_chunks = pdf_loader.load_and_split(text_splitter)

In [7]:
len(tesla_10k_chunks)

3337

In [8]:
print(tesla_10k_chunks[0].page_content)

UNITED	STATES
SECURITIES	AND	EXCHANGE	COMMISSION
Washington,	D.C.	20549
	
FORM	
10-K/A
(Amendment	No.	1)
	
(Mark	One)
☒
ANNUAL	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	fiscal	year	ended	
December	31,	
2021
	
OR
☐
TRANSITION	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	transition	period	from	
																				
	to	
																					
Commission	File	Number:	
001-34756
	
Tesla,	Inc.
(Exact	name	of	registrant	as	specified	in	its	charter)
	
	
Delaware
	
91-2197729
(State	or	other	jurisdiction	of
incorporation	or	organization)
	
(I.R.S.	Employer
Identification	No.)
	
	
	
1	Tesla	Road
Austin
,	
Texas
	
78725
(Address	of	principal	executive	offices)
	
(Zip	Code)
(
512
)	
516-8177
(Registrant’s	telephone	number,	including	area	code)
Securities	registered	pursuant	to	Section	12(b)	of	the	Act:
	
Title	of	each	class
Trading	Symbol(s)
Name	of	each	exchange	on	which	registered
Common	stock
TSLA
The	Nasdaq	Gl

In [36]:
tesla_10k_collection = 'tesla-10k-2019-to-2023'

In [33]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

C:\Users\Rohit kumar\AppData\Local\Temp\ipykernel_8240\726496232.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")


In [37]:
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [38]:
chromadb_client.count_collections()

2

In [45]:
vectorstore = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [46]:
chromadb_client.list_collections()

['hypothetical_questions', 'tesla-10k-2019-to-2023']

In [47]:
print(tesla_10k_chunks[0].page_content)

UNITED	STATES
SECURITIES	AND	EXCHANGE	COMMISSION
Washington,	D.C.	20549
	
FORM	
10-K/A
(Amendment	No.	1)
	
(Mark	One)
☒
ANNUAL	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	fiscal	year	ended	
December	31,	
2021
	
OR
☐
TRANSITION	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	transition	period	from	
																				
	to	
																					
Commission	File	Number:	
001-34756
	
Tesla,	Inc.
(Exact	name	of	registrant	as	specified	in	its	charter)
	
	
Delaware
	
91-2197729
(State	or	other	jurisdiction	of
incorporation	or	organization)
	
(I.R.S.	Employer
Identification	No.)
	
	
	
1	Tesla	Road
Austin
,	
Texas
	
78725
(Address	of	principal	executive	offices)
	
(Zip	Code)
(
512
)	
516-8177
(Registrant’s	telephone	number,	including	area	code)
Securities	registered	pursuant	to	Section	12(b)	of	the	Act:
	
Title	of	each	class
Trading	Symbol(s)
Name	of	each	exchange	on	which	registered
Common	stock
TSLA
The	Nasdaq	Gl

In [ ]:
i = 0 # Initialize the starting index for the chunks

while i < len(tesla_10k_chunks): # Iterate while the index is less than the total number of chunks
    vectorstore.add_documents( # Add documents to the vector store in batches of 500
        documents=tesla_10k_chunks[i:i+500], # Get the current batch of 500 chunks
        ids=["text_" + str(i) for i in range(i, i+500)] # Assign unique IDs to each chunk in the batch
    )

    i += 500 # Increment the index by 500 to move to the next batch
    time.sleep(5) # Pause for 5 seconds to avoid rate limiting issues with the vector store

In [38]:
import chromadb

from langchain_chroma import Chroma



In [39]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [40]:
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001AD47CD7390>, search_kwargs={'k': 5})

In [41]:
user_query = "What was the automotive revenue in 2021?"

## Hypothetical Questions

In [11]:
from dotenv import load_dotenv
load_dotenv()
import os
os.environ['NVIDIA_API_KEY'] = os.getenv('NVIDIA_API_KEY')

In [12]:
from openai import OpenAI
import os


In [13]:
client = OpenAI(
    api_key=os.environ["NVIDIA_API_KEY"],
    base_url="https://integrate.api.nvidia.com/v1"
)

In [14]:
model_name = "meta/llama-3.1-70b-instruct"

In [15]:

user_message_template = """
<Document>
{document}
</Document>
"""

In [16]:
tesla_10k_chunks

[Document(metadata={'producer': 'Qt 5.11.3', 'creator': 'wkhtmltopdf 0.12.5', 'creationdate': '2022-05-02T10:10:26+00:00', 'title': '', 'source': 'tesla-annual-reports\\tsla-10ka_20211231-gen.pdf', 'total_pages': 56, 'page': 0, 'page_label': '1'}, page_content='UNITED\tSTATES\nSECURITIES\tAND\tEXCHANGE\tCOMMISSION\nWashington,\tD.C.\t20549\n\t\nFORM\t\n10-K/A\n(Amendment\tNo.\t1)\n\t\n(Mark\tOne)\n☒\nANNUAL\tREPORT\tPURSUANT\tTO\tSECTION\t13\tOR\t15(d)\tOF\tTHE\tSECURITIES\tEXCHANGE\tACT\tOF\t1934\nFor\tthe\tfiscal\tyear\tended\t\nDecember\t31,\t\n2021\n\t\nOR\n☐\nTRANSITION\tREPORT\tPURSUANT\tTO\tSECTION\t13\tOR\t15(d)\tOF\tTHE\tSECURITIES\tEXCHANGE\tACT\tOF\t1934\nFor\tthe\ttransition\tperiod\tfrom\t\n\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\n\tto\t\n\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\nCommission\tFile\tNumber:\t\n001-34756\n\t\nTesla,\tInc.\n(Exact\tname\tof\tregistrant\tas\tspecified\tin\tits\tcharter)\n\t\n\t\nDelaware\n\t\n91-2197729\n(State\tor\tother\tjurisdiction\tof\n

In [54]:
batch_questions

[{'batch_id': 1,
  'questions': "What is the composition of Tesla's Board of Directors and their respective backgrounds and qualifications?\nHow does Tesla's compensation philosophy and program align with its mission to accelerate the world's transition to sustainable energy?\nWhat are the key terms and conditions of Elon Musk's 2018 CEO Performance Award, including the vesting schedule and potential payouts?",
  'context': 'UNITED\tSTATES\nSECURITIES\tAND\tEXCHANGE\tCOMMISSION\nWashington,\tD.C.\t20549\n\t\nFORM\t\n10-K/A\n(Amendment\tNo.\t1)\n\t\n(Mark\tOne)\n☒\nANNUAL\tREPORT\tPURSUANT\tTO\tSECTION\t13\tOR\t15(d)\tOF\tTHE\tSECURITIES\tEXCHANGE\tACT\tOF\t\n\nYes\n\t\t\n☒\n\t\t\t\tNo\t\t\n☐\nIndicate\tby\tcheck\tmark\tif\tthe\tregistrant\tis\tnot\trequired\tto\tfile\treports\tpursuant\tto\tSection\t13\tor\t15(d)\tof\tthe\tAct.\t\t\t\tYes\t\t\n☐\n\t\t\t\t\nNo\n\t\t\n☒\nIndicate\tby\tcheck\tmark\twhether\tthe\tregi\n\nEmerging\tgrowth\tcompany\n\t\n☐\n\t\n\t\n\t\n\t\nIf\tan\temerging\tg

In [19]:
BATCH_SIZE = 100

chunk_batches = []

for i in range(0, len(tesla_10k_chunks), BATCH_SIZE):

    batch = tesla_10k_chunks[i:i+BATCH_SIZE]

    # Token limit control
    batch_text = "\n\n".join(
        [doc.page_content[:200] for doc in batch]
    )

    chunk_batches.append(batch_text)

print(f"Total batches: {len(chunk_batches)}")


def generate_hypothetical_questions(batch_text):

    response = client.chat.completions.create(
        model=model_name,
        temperature=0,
        max_tokens=300,
        messages=[
            {
                "role": "system",
                "content": """
You will receive multiple document chunks.

Generate ONLY 3 hypothetical questions that best represent
the information across ALL chunks.

Rules:
- Generate exactly 3 questions.
- One question per line.
- No numbering.
- No explanations.
"""
            },
            {
                "role": "user",
                "content": user_message_template.format(
                    document=batch_text
                )
            }
        ]
    )

    return response.choices[0].message.content

user_query = "What are key insights from Tesla annual report?"

retrieved_HQS = []

for idx, batch_text in enumerate(chunk_batches):

    try:

        questions_raw = generate_hypothetical_questions(batch_text)

        # Convert string → list
        questions = [q.strip() for q in questions_raw.split("\n") if q.strip()]

        for q in questions:
            retrieved_HQS.append({
                "hypothetical_question": q,
                "parent_chunk_id": f"chunk_{idx+1}",
                "section": "unknown",
                "year": 2025,
                "score": 0.5
            })

        print(f"Processed Batch {idx + 1}")

    except Exception as e:
        print(f"Batch {idx + 1} failed: {e}")

Total batches: 34
Processed Batch 1
Processed Batch 2
Processed Batch 3
Processed Batch 4
Processed Batch 5
Processed Batch 6
Processed Batch 7
Processed Batch 8
Processed Batch 9
Processed Batch 10
Processed Batch 11
Processed Batch 12
Processed Batch 13
Processed Batch 14
Processed Batch 15
Processed Batch 16
Processed Batch 17
Processed Batch 18
Batch 19 failed: Error code: 429 - {'status': 429, 'title': 'Too Many Requests'}
Processed Batch 20
Processed Batch 21
Processed Batch 22
Processed Batch 23
Batch 24 failed: Error code: 429 - {'status': 429, 'title': 'Too Many Requests'}
Processed Batch 25
Batch 26 failed: Error code: 429 - {'status': 429, 'title': 'Too Many Requests'}
Processed Batch 27
Processed Batch 28
Processed Batch 29
Batch 30 failed: Error code: 429 - {'status': 429, 'title': 'Too Many Requests'}
Processed Batch 31
Processed Batch 32
Processed Batch 33
Processed Batch 34


In [21]:
len(retrieved_HQS)

90

In [28]:
retrieved_HQS = retrieved_HQS[:10]

In [29]:
parent_chunks = []

for idx, batch_text in enumerate(chunk_batches):
    parent_chunks.append({
        "chunk_id": f"chunk_{idx+1}",
        "source_doc": "tesla_10k",
        "section": "unknown",
        "year": 2025
    })

In [30]:
import json

In [27]:
output = {
    "question_id": "HQ1",
    "user_query": user_query,
    "retrieved_hypothetical_questions": retrieved_HQS,
    "parent_chunks_used": parent_chunks,
    "final_answer": "This summarizes key insights from Tesla reports.",
    "citations": [],
    "comparison_with_baseline": "Hypothetical questions improve retrieval relevance."
}

with open("final_output.json", "w") as f:
    json.dump(output, f, indent=4)

print("✅ JSON file created")

✅ JSON file created


In [49]:
hypothetical_questions_vectorstore = Chroma(
    collection_name="retrieved_HQS",
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [50]:
chromadb_client.list_collections()

['retrieved_HQS', 'hypothetical_questions', 'tesla-10k-2019-to-2023']

In [53]:
from langchain_core.documents import Document

hypothetical_documents = []

for item in retrieved_HQS:

    hypothetical_documents.append(
        Document(
            page_content=item["hypothetical_question"],
            metadata={
                "batch_id": item["parent_chunk_id"]
            }
        )
    )

In [54]:
print(type(hypothetical_documents[0]))

<class 'langchain_core.documents.base.Document'>


In [55]:
hypothetical_questions_vectorstore.add_documents(
    documents=hypothetical_documents)

['0b87c13b-6e93-47ce-b1b8-80a2250e5075',
 'e8575c2d-aeff-49f6-8c9c-b4309b58fd7d',
 'e546d25a-8aa9-41dd-a47e-f111b6e5e882',
 'ff5639d0-28d1-458d-9b0d-bed2dfeb1c55',
 '44e952e1-76f0-4c98-bf72-efeb6f7b4415',
 'ec29dcb5-e8e4-4780-811a-2e8b43035c42',
 '15db1479-54ff-4fdd-af89-52c2253f815c',
 '49665d19-8470-47a7-a6da-fb875e784b19',
 '56504b22-b2f3-4f0c-9647-2ca2f57a878d',
 'c018f609-b75c-4433-b3dd-cbd1a1847231']

In [56]:
retriever = hypothetical_questions_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={'k': 5}
)

In [57]:
hypothetical_questions_retrieved = retriever.invoke(user_query)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [58]:
hypothetical_questions_retrieved

[Document(id='49665d19-8470-47a7-a6da-fb875e784b19', metadata={'batch_id': 'chunk_3'}, page_content="How does Tesla's business model, including its focus on electric vehicles and energy storage, impact its financial performance and growth prospects?"),
 Document(id='ff5639d0-28d1-458d-9b0d-bed2dfeb1c55', metadata={'batch_id': 'chunk_2'}, page_content="What types of products and services does Tesla offer, and how do they contribute to the company's overall mission?"),
 Document(id='ec29dcb5-e8e4-4780-811a-2e8b43035c42', metadata={'batch_id': 'chunk_2'}, page_content="What regulatory and environmental factors does Tesla need to consider in its business operations, and how do these factors impact the company's products and services?"),
 Document(id='15db1479-54ff-4fdd-af89-52c2253f815c', metadata={'batch_id': 'chunk_3'}, page_content='What are the key risks and challenges that Tesla faces in its business operations, including regulatory constraints, competition, and supply chain disruptio

In [59]:
print(hypothetical_questions_retrieved[0].metadata)

{'batch_id': 'chunk_3'}


In [62]:
BATCH_SIZE = 100

retrieved_documents = []

for doc in hypothetical_questions_retrieved:

    batch_id = doc.metadata["batch_id"]
    # Extract number from "chunk_X" format
    batch_num = int(batch_id.split("_")[1])

    start_idx = (batch_num - 1) * BATCH_SIZE
    end_idx = min(
        start_idx + BATCH_SIZE,
        len(tesla_10k_chunks)
    )

    retrieved_documents.extend(
        tesla_10k_chunks[start_idx:end_idx]
    )

print(f"Retrieved {len(retrieved_documents)} chunks")

Retrieved 500 chunks


In [63]:
print(retrieved_documents)

[Document(metadata={'producer': 'Qt 5.11.3', 'creator': 'wkhtmltopdf 0.12.5', 'creationdate': '2020-09-30T05:19:35+00:00', 'title': '', 'source': 'tesla-annual-reports\\tsla-10k_20191231-gen_0.pdf', 'total_pages': 297, 'page': 15, 'page_label': '16'}, page_content='regulatory\tconstraints\tover\twhich\twe\thave\tno\tcontrol,\tour\tgoal\tis\ta\tfully\tautonomously-driven\tfuture\tthat\timproves\tsafety\tand\tprovides\tour\tcustomers\nwith\tconvenience\tand\tadditional\tincome\tthrough\tparticipation\tin\tan\tautonomous\tTesla\tride-hailing\tnetwork.\tThis\tnetwork,\twhich\twill\talso\tinclude\tour\nown\tfleet\tof\tvehicles,\twill\talso\tallow\tus\tto\taccess\ta\tnew\tcustomer\tbase\teven\tas\tmodes\tof\ttransportation\tevolve.\tFinally,\tour\tvehicles\toffer\tunparalleled\nin-vehicle\tentertainment\tfeatures,\tcurrently\tincluding\tInternet\tsearch,\tmusic\tservices,\tpassenger\tkaraoke,\tand\tparked\tvideo\tstreaming\tand\tgaming.\n13'), Document(metadata={'producer': 'Qt 5.11.3', 'cre

In [64]:
print(type(retrieved_documents))

<class 'list'>


In [75]:
print(type(retrieved_documents[0]))

<class 'langchain_core.documents.base.Document'>


In [65]:
retrieved_context = "\n\n".join(
    doc.page_content
    for doc in retrieved_documents
)

In [66]:
retrieved_context

"regulatory\tconstraints\tover\twhich\twe\thave\tno\tcontrol,\tour\tgoal\tis\ta\tfully\tautonomously-driven\tfuture\tthat\timproves\tsafety\tand\tprovides\tour\tcustomers\nwith\tconvenience\tand\tadditional\tincome\tthrough\tparticipation\tin\tan\tautonomous\tTesla\tride-hailing\tnetwork.\tThis\tnetwork,\twhich\twill\talso\tinclude\tour\nown\tfleet\tof\tvehicles,\twill\talso\tallow\tus\tto\taccess\ta\tnew\tcustomer\tbase\teven\tas\tmodes\tof\ttransportation\tevolve.\tFinally,\tour\tvehicles\toffer\tunparalleled\nin-vehicle\tentertainment\tfeatures,\tcurrently\tincluding\tInternet\tsearch,\tmusic\tservices,\tpassenger\tkaraoke,\tand\tparked\tvideo\tstreaming\tand\tgaming.\n13\n\nEnergy\tGeneration\tand\tStorage\nEnergy\tStorage\tSystems\nThe\tmarket\tfor\tenergy\tstorage\tproducts\tis\talso\thighly\tcompetitive.\tEstablished\tcompanies,\tsuch\tas\tAES\tEnergy\tStorage,\tSiemens,\tLG\tChem\tand\nSamsung,\tas\twell\tas\tvarious\temerging\tcompanies,\thave\tintroduced\tproducts\tthat\tare\

In [67]:
len(retrieved_context)

646193

In [79]:
qna_system_message1 = """
You are a helpful AI assistant.

Answer the user's question using only the information provided in the context.

Rules:
- Use only the provided context.
- Do not make up information.
- If the answer is not available in the context, say:
  "I could not find the answer in the provided context."
- Be concise and accurate.
"""
qna_user_message_template1 = """
Context:
{context}

Question:
{question}

Answer:
"""

final_prompt = [
    {
        "role": "system",
        "content": qna_system_message1
    },
    {
        "role": "user",
        "content": qna_user_message_template1.format(
            context=retrieved_context,
            question=user_query
        )
    }
]

response = client.chat.completions.create(
    model=model_name,
    messages=final_prompt,
    temperature=0
)

final_answer = response.choices[0].message.content
print(final_answer)

$ 67,210
